In [ ]:
!pip install ollama

In [5]:
import ollama

response = ollama.chat(
    model="llama3.1:8B",
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user",   "content": "Why is the sky blue?"},
    ],
)
print(response.message.content)


The sky appears blue because of a phenomenon called Rayleigh scattering, where shorter (blue) wavelengths of light are scattered more than longer (red) wavelengths by the tiny molecules of gases in the atmosphere.


In [4]:

# Streaming: tokens appear as they're generated
for chunk in ollama.chat(
    model="llama3.1:8B",
    messages=[{"role": "user", "content": "Write a haiku about Python."}],
    stream=True,
):
    print(chunk.message.content, end="", flush=True)

Here is a haiku about Python:

Code flows like a stream
Indented and serene beauty
Syntax is its song

In [ ]:
!pip install google-genai python-dotenv

In [6]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
from google import genai

client = genai.Client()  # picks up GEMINI_API_KEY

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Explain how AI works in a few words",
)
print(response.text)

AI works by analyzing massive amounts of **data** to find **patterns**, which it uses to make **predictions** or decisions. 

In short: **It learns from data to spot patterns and solve problems.**


In [ ]:
# langchain            -> the agent framework (create_agent, @tool)
# langchain-ollama     -> lets LangChain talk to local Ollama models
# langchain-google-genai -> lets LangChain talk to Gemini (cloud)
!pip install -U langchain langchain-ollama langchain-google-genai

In [8]:
from langchain.tools import tool

# A tool is just a normal Python function with the @tool decorator.
# The model never sees the code. It only sees the function NAME, the
# argument TYPES and the DOCSTRING, so write the docstring like a prompt:
# it is how the model decides WHEN to use the tool.

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    # Demo data. Replace with a real weather API call.
    # This is what lets the LLM reach outside its training data.
    fake = {"delhi": "34°C, hazy", "london": "14°C, rain", "tokyo": "22°C, clear"}
    return fake.get(city.lower(), "no data for that city")

@tool
def calculate(expression: str) -> str:
    """Evaluate a basic math expression like '23 * 7 + 4'."""
    # LLMs are unreliable at exact arithmetic, so we hand the maths to Python.
    # Only allow digits and operators before calling eval (demo-level safety;
    # never eval raw model output in real code).
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return "Only numbers and + - * / ( ) are allowed."
    return str(eval(expression))

In [9]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

# The "brain": a local model served by Ollama.
# It must support tool calling (llama3.1 does). temperature=0 keeps
# tool choices consistent between runs, which is good for a demo.
model = ChatOllama(model="llama3.1:8B", temperature=0)

# create_agent wires brain + tools + the loop together:
#   model decides -> tool runs -> result goes back to model -> repeat
agent = create_agent(
    model,
    tools=[get_weather, calculate],  # the "hands": what the agent is allowed to do
    system_prompt="You are a helpful assistant. Use tools when they help.",
)

# One question that needs TWO different tools, so the agent has to loop.
result = agent.invoke({
    "messages": [{"role": "user", "content": "What's the weather in Delhi, and what is 23 * 7 + 4?"}]
})

# Print the whole trace. Read it top to bottom:
#   Human message -> AI message (tool calls) -> Tool messages (results) -> final AI answer
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

What's the weather in Delhi, and what is 23 * 7 + 4?
================================== Ai Message ==================================
Tool Calls:
  get_weather (b71268d0-4acb-4d69-90c3-4d6f11527663)
 Call ID: b71268d0-4acb-4d69-90c3-4d6f11527663
  Args:
    city: Delhi
  calculate (39b31df8-5265-43c1-aa52-efaafdb5251a)
 Call ID: 39b31df8-5265-43c1-aa52-efaafdb5251a
  Args:
    expression: 23 * 7 + 4
================================= Tool Message =================================
Name: get_weather

34°C, hazy
================================= Tool Message =================================
Name: calculate

165
================================== Ai Message ==================================

The weather in Delhi is 34°C, hazy. The result of the calculation 23 * 7 + 4 is 165.
